# Model training quick reference

For the complete Task 1 submission—including preprocessing, exploratory analysis, six question-specific visualizations and interpretations, chronological hyperparameter tuning, and final evaluation—run [Task1_Time_Series_Analysis.ipynb](Task1_Time_Series_Analysis.ipynb).

This shorter notebook is retained as a training walkthrough. It does not replace the full Task 1 analysis.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

PROJECT_ROOT = Path.cwd()
DATASET_PATH = PROJECT_ROOT / 'dataset' / 'household_power_consumption.txt'
MODELS_DIR = PROJECT_ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_PATH.is_file():
    raise FileNotFoundError(f'Dataset not found: {DATASET_PATH}')

## Load and resample

The file uses semicolons and `?` for missing values. Aggregate numeric measurements to hourly means and interpolate missing hourly values.

In [ ]:
df = pd.read_csv(DATASET_PATH, sep=';', na_values=['?'], low_memory=False)
df['Datetime'] = pd.to_datetime(
    df['Date'] + ' ' + df['Time'],
    format='%d/%m/%Y %H:%M:%S'
)
df = df.set_index('Datetime').drop(columns=['Date', 'Time'])

for column in df.columns:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Leave missing hours missing for modeling; incomplete lag windows are dropped.
df_hourly = df.resample('h').mean()
df_hourly.head()

## Engineer leakage-safe features

Voltage and intensity enter only as lagged values. Shift the 24-hour rolling mean so it excludes the target hour.

In [ ]:
TARGET = 'Global_active_power'
feature_columns = [
    'power_lag_1h',
    'power_lag_24h',
    'voltage_lag_1h',
    'intensity_lag_1h',
    'rolling_mean_24h',
]

df_hourly['power_lag_1h'] = df_hourly[TARGET].shift(1)
df_hourly['power_lag_24h'] = df_hourly[TARGET].shift(24)
df_hourly['voltage_lag_1h'] = df_hourly['Voltage'].shift(1)
df_hourly['intensity_lag_1h'] = df_hourly['Global_intensity'].shift(1)
df_hourly['rolling_mean_24h'] = (
    df_hourly[TARGET].rolling(window=24).mean().shift(1)
)
df_hourly = df_hourly.dropna(subset=feature_columns + [TARGET])

X = df_hourly[feature_columns]
y = df_hourly[TARGET]
print(f'Rows available: {len(df_hourly):,}')
print(f'Features in model order: {feature_columns}')

## Chronological split, train, and evaluate

Use the first 80% of rows for training and the final 20% for evaluation, without shuffling.

In [ ]:
train_size = int(len(df_hourly) * 0.8)
if train_size < 1 or train_size >= len(df_hourly):
    raise ValueError('Not enough hourly rows for an 80/20 chronological split.')

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train, y_test = y.iloc[:train_size], y.iloc[train_size:]

model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
print(f'Test RMSE: {rmse:.4f}')
print(f'Test MAE: {mae:.4f}')

## Export the model and feature metadata

Save the trained model and exact feature names in training order.

In [ ]:
model_path = MODELS_DIR / 'model.joblib'
joblib.dump(model, model_path)

metadata = {
    'feature_names': feature_columns,
    'target': TARGET,
}
metadata_path = MODELS_DIR / 'model_metadata.json'
with metadata_path.open('w', encoding='utf-8') as metadata_file:
    json.dump(metadata, metadata_file, indent=4)
    metadata_file.write('\n')

print(f'Model saved to {model_path}')
print(f'Metadata saved to {metadata_path}')